In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sys
import math
import os
import glob 
from collections import defaultdict

On boucle sur les transects

In [2]:
prov_folder = sorted(glob.glob("/data/rd_exchange/salbernhe/provinces/netcdf_monthly_provinces/provinces_Y*_M*.nc"))

PATH_prov = Path("/data/rd_exchange/salbernhe/provinces/netcdf_monthly_provinces/provinces_Y1998_M01.nc")
ds_prov = xr.open_dataset(PATH_prov)
ds_prov

<xarray.Dataset> Size: 35MB
Dimensions:    (latitude: 2040, longitude: 4320)
Coordinates:
  * latitude   (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.75 89.83 89.92
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    province   (latitude, longitude) float32 35MB ...
Attributes:
    title:        Regions biome T, Str, npp (from GLORYS/VGPM)
    institution:  CLS
    references:   http://www.cls.fr

In [3]:
PATH_sv = Path("/data/rd_exchange/salbernhe/data/acoustics/profiles38kHz10t0750m20251006.nc")
ds_sv = xr.open_dataset(PATH_sv)
ds_sv

<xarray.Dataset> Size: 351MB
Dimensions:     (file_index: 318, time: 541061, range: 75)
Coordinates:
  * file_index  (file_index) int64 3kB 1 2 3 4 5 6 7 ... 313 314 315 316 317 318
  * time        (time) datetime64[ns] 4MB 2008-04-09T10:00:34.000003072 ... 2...
  * range       (range) int64 600B 10 20 30 40 50 60 ... 700 710 720 730 740 750
Data variables:
    file_label  (file_index) <U116 148kB ...
    longitude   (time) float64 4MB ...
    latitude    (time) float64 4MB ...
    file        (time) int64 4MB ...
    interval    (time) float64 4MB ...
    sunangle    (time) float64 4MB ...
    Sv          (range, time) float64 325MB ...
    frequency   |S1 1B ...

In [ ]:
print(ds_sv['file_label'].sel(file_index=3).item())
##A regarder, les dates n'ont pas toute le même format: regarder avec datetime

BAS_SONA_ShipResearch_RRSJamesClarkRoss_M_AtlanticOcean_2014-10-03T14Z_2014-10-17T23Z.nc


In [ ]:
#fonction pour passer de db à linéaire et vice-versa

def db_to_lin(x):
    return 10.0 ** (x / 10.0)

def lin_to_db(x):
    return 10.0 * np.log10(x)


end_dawn = 6      # au-dessus de +6°: jour
end_dusk = -6    #en dessous de -6°: nuit

rows_day = []
rows_night=[]

In [ ]:


def process_segment(ds_segment, province, file_idx): #calcule les moyennes d'un transect jour/nuit d'un segment et alimente row[]

    ds_sv_lin = db_to_lin(ds_segment["Sv"])

    is_day = ds_segment["sunangle"] > end_dawn
    is_night = ds_segment["sunangle"]< end_dusk

    range = ds_segment["range"].values

    if is_day.sum()>0: #On fait la somme des Trues(1) et des Falses(0) des booléens pour vérifier qu'on a bien des valeurs pour le jour
        sv_day = lin_to_db(ds_sv_lin.isel(time=is_day).mean(dim="time", skipna=True)).values

        for r, sv_val in zip(range, sv_day):
            rows_day.append({"file_idx": file_idx, "province": province, "range": r, "sv": sv_val})

    if is_night.sum() > 0: #idem, on vérifie qu'on a bien des valeurs pour la nuit. 
        sv_night = lin_to_db(ds_sv_lin.isel(time=is_night).mean(dim="time", skipna=True)).values

        for r, sv_val in zip(range, sv_night):
            rows_night.append({"file_idx": file_idx,"province": province, "range": r, "sv": sv_val})



for file_idx_da in ds_sv["file_index"]: #on boucle sur les index du ds_sv

    file_idx = file_idx_da.item() #pour passer d'un array à juste un int Python, mais à enlever plus tard si la ligne est inutile
    file_label = ds_sv["file_label"].sel(file_index=file_idx).item() #.item() parce que sinon renvoie un dataArray 0-dimensionnel, et pas juste une sttring comme je veux
    file_str = str(file_label) #on a l'interieur du array 0-dim, on le passe en format str
    date_part = file_str.split("_")[-1]  #ici récupère seulement la dernière date, il faudrait faire en sorte de prendre celle avec le plus de données
    year = date_part[:4]     
    month = date_part[5:7] 

    wanted_file_name = f"provinces_Y{year}_M{month}.nc"
    matching_file_prov = None
    for file_prov in prov_folder: 
        if os.path.basename(file_prov)== wanted_file_name:
            matching_file_prov = file_prov
            break

    if matching_file_prov is None:
        print(f"Aucune correspondance trouvée pour {file_str} ")

    
    #On a donc la correspondance entre date du transect et date des provinces. Maintenant on cherche à associer chaque partie du transect à la province qu'il traverse (via un mask)

    ds_prov = xr.open_dataset(matching_file_prov)
    
    mask = ds_sv["file"] == file_idx # tableau bouléen qui parcours tous les pings, et affiche True si le ping appartient au fichier qu'on traite actuellement. 
    ds_file = ds_sv.where(mask,drop=True)
    lat_ping = ds_sv["latitude"].where(mask, drop=True)
    lon_ping = ds_sv["longitude"].where(mask, drop=True)
    
    list_prov_with_NaN = ds_prov["province"].sel(latitude=xr.DataArray(lat_ping.values, dims="time"), longitude= xr.DataArray(lon_ping.values, dims="time"), method="nearest").values
    list_prov = np.unique(list_prov_with_NaN[~np.isnan(list_prov_with_NaN)]) #liste propre des provinces sans les Nans




#cas 1. le transect ne traverse qu'une province
    if len(list_prov) == 1:
        province = list_prov[0]
        process_segment(ds_file, province, file_idx)

#Cas 2. Le transect traverse plusieurs provinces

    else:
        idx_change_points = np.where(np.diff(list_prov_with_NaN) != 0)[0] +1 #contient les indices où la province change
        segments_idx = np.split(np.arange(len(list_prov_with_NaN)), idx_change_points) 
       

        for seg in segments_idx:
            list_prov_seg = list_prov_with_NaN[seg[0]]
            if np.isnan(list_prov_seg):
                continue #si le segment est hors de la grille on l'ignore
        
           
            ds_segment = ds_file.isel(time=seg)
            process_segment(ds_segment, list_prov_seg, file_idx)
            
           
    ds_prov.close()


df_day = pd.DataFrame(rows_day)
df_night = pd.DataFrame(rows_night)   

Aucune correspondance trouvée pour IMOS_SOOP-BA_AE_20080131T080942Z_ZMFR_FV02_Tangaroa-EK60-38_END-20080209T063750Z_C-20160405T221754Z.nc 


ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [12]:
print(ds_sv["sunangle"].min().item(), ds_sv["sunangle"].max().item())

-88.95622222222222 89.9166388888889
